<div style="background: linear-gradient(135deg, #1b4332 0%, #d62828 60%, #e76f51 100%); padding: 35px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; font-family: 'Segoe UI', sans-serif; font-size: 2.2em; margin: 0; font-weight: 700;">🤖 Experimentos de Modelado Predictivo</h1>
  <p style="color: #ffd6cc; font-size: 1.1em; margin: 12px 0 0;">E-Commerce Churn Prediction &nbsp;|&nbsp; No Country — Equipo 40</p>
</div>

> **Objetivo:** Entrenar, comparar y seleccionar el mejor clasificador binario para detectar churn.  
> **Métrica prioritaria:** **Recall** — minimizar falsos negativos es más importante que evitar falsos positivos.  
> **Estrategia de validación:** Cross-validation estratificada 5-Fold para estimaciones robustas.

---

## 📋 Tabla de Contenidos

| # | Sección | Descripción |
|---|---------|-------------|
| 1 | [⚙️ Configuración](#1) | Setup y carga de dependencias |
| 2 | [📥 Datos y Features](#2) | Preparación del dataset de modelado |
| 3 | [🚀 Entrenamiento CV](#3) | Entrenamiento con validación cruzada |
| 4 | [📊 Comparativa de Modelos](#4) | Recall, F1, AUC comparados visualmente |
| 5 | [🎯 Evaluación Final en Test](#5) | Métricas en el conjunto de test holdout |
| 6 | [📈 Análisis del Threshold](#6) | Precision-Recall vs umbral de decisión |
| 7 | [💾 Guardar Champion](#7) | Registry del modelo + metrics.md |
| 8 | [✅ Conclusiones](#8) | Hallazgos y decisión técnica |

---
## ⚙️ 1. Configuración <a id='1'></a>

In [ ]:
import sys, os
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
os.environ['DISABLE_PANDERA_IMPORT_WARNING'] = 'True'

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'primary':'#2d6a4f','secondary':'#52b788','accent':'#1a759f','warning':'#e76f51','danger':'#d62828'}

with open('config/config.yaml') as f:
    cfg = yaml.safe_load(f)

for d in ['reports/figures/modeling', 'models/champion', 'models/registry']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✅ Entorno configurado')

---
## 📥 2. Carga y Preparación de Datos <a id='2'></a>

> Se carga el dataset enriquecido con todas las features generadas en los notebooks anteriores.

> **⚠️ Importante sobre el desbalance de clases:**  
> Cuando `Churn=1` es la clase minoritaria, un modelo naive que predice siempre `0` obtendría alta *accuracy* pero **Recall = 0%** — detectaría CERO churners.  
> Por eso usamos `class_weight='balanced'` y priorizamos **Recall** como métrica de selección.

In [ ]:
rfm = pd.read_parquet('data/processed/rfm_with_churn.parquet')

base_features    = cfg['features']['rfm']
advanced_features = [f for f in cfg['features']['advanced'] if f in rfm.columns]
FEATURES = base_features + advanced_features
TARGET   = 'CHURN'

print(f'✅ Dataset cargado: {len(rfm):,} clientes')
print(f'📋 Features seleccionadas ({len(FEATURES)}): {FEATURES}')
print(f'🎯 Target: {TARGET}')

churn_n   = rfm[TARGET].sum()
active_n  = (rfm[TARGET]==0).sum()
ratio     = active_n / churn_n if churn_n > 0 else float('inf')

print(f'\n   Churn = 1 : {churn_n:>5,} clientes ({rfm[TARGET].mean():.1%})')
print(f'   Churn = 0 : {active_n:>5,} clientes ({1-rfm[TARGET].mean():.1%})')
print(f'   Ratio     : {ratio:.2f}:1 → {'manejable con class_weight' if ratio < 10 else 'requiere técnicas adicionales'}')

In [ ]:
from src.features.scaler import FeatureScaler

features_available = [f for f in FEATURES if f in rfm.columns]
scaler = FeatureScaler(features=features_available)
rfm_scaled = scaler.fit_transform(rfm)
scaler.save('models/champion/scaler.pkl')
print(f'✅ Features escaladas y scaler guardado → models/champion/scaler.pkl')

---
## 🚀 3. Entrenamiento con Validación Cruzada <a id='3'></a>

> **Estrategia:** `StratifiedKFold` con 5 folds garantiza que cada fold mantenga la misma proporción de churners.  
> El `ModelTrainer` entrena automáticamente **3 modelos** en paralelo:

| Modelo | Ventaja para Churn | Config de desbalance |
|--------|--------------------|---------------------|
| **Random Forest** | Robusto a outliers, feature importance nativa | `class_weight='balanced'` |
| **Logistic Regression** | Probabilidades bien calibradas, interpretable | `class_weight='balanced'` |
| **XGBoost** | Estado del arte en datos tabulares, gradientes adaptativos | `scale_pos_weight` dinámico |

In [ ]:
from src.modeling.trainer import ModelTrainer

model_cfg = cfg['modeling']
trainer = ModelTrainer(
    features=features_available,
    target=TARGET,
    test_size=model_cfg['test_size'],
    cv_folds=model_cfg['cv_folds'],
    random_state=model_cfg['random_state'],
    primary_metric=model_cfg['primary_metric']
)

print('🚀 Entrenando modelos con validación cruzada...')
print(f'   CV folds    : {model_cfg["cv_folds"]}')
print(f'   Random seed : {model_cfg["random_state"]}')
print(f'   Métrica     : {model_cfg["primary_metric"].upper()}')
print()
trainer.fit(rfm_scaled)

print(f'\n🏆 Modelo seleccionado: {trainer.best_model_name}')

---
## 📊 4. Comparativa de Resultados CV <a id='4'></a>

> Los resultados de la validación cruzada son la estimación más confiable del rendimiento real del modelo,  
> ya que aprovechan todos los datos para evaluación sin introducir data leakage.

In [ ]:
cv_df = pd.DataFrame(trainer.cv_results).T.reset_index()
cv_df.columns = ['Model','Recall','Recall_Std','F1','AUC','Accuracy']
cv_df = cv_df.sort_values('Recall', ascending=False)

print('=== Resultados de Validación Cruzada 5-Fold ===')
print(cv_df.round(4).to_string(index=False))

print(f'\n🏆 Mejor Recall: {cv_df.iloc[0]["Recall"]:.4f} ({cv_df.iloc[0]["Model"]})')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('📊 Comparativa de Modelos — Validación Cruzada 5-Fold', fontsize=13, fontweight='bold')

metrics = ['Recall','F1','AUC','Accuracy']
m_colors = [COLORS['danger'], COLORS['primary'], COLORS['accent'], COLORS['secondary']]
m_labels = ['⭐ Recall\n(prioritaria)','F1 Score','ROC-AUC','Accuracy']

for ax, metric, color, label in zip(axes, metrics, m_colors, m_labels):
    bars = ax.bar(cv_df['Model'], cv_df[metric], color=color, alpha=0.85, edgecolor='white')
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(cv_df['Model'], rotation=25, ha='right', fontsize=8)
    for bar, val in zip(bars, cv_df[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.savefig('reports/figures/modeling/model_comparison_cv.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Guardado: reports/figures/modeling/model_comparison_cv.png')

---
## 🎯 5. Evaluación Final en Test Set <a id='5'></a>

> El test set es un **conjunto de holdout** que el modelo nunca vio durante entrenamiento ni tuning.  
> Es la estimación más honesta del rendimiento en producción.

In [ ]:
from src.modeling.evaluator import ModelEvaluator

evaluator = ModelEvaluator(figures_dir='reports/figures/modeling')
test_metrics = evaluator.evaluate(
    trainer.best_model, trainer.best_model_name,
    trainer.X_test, trainer.y_test
)

print(f'=== Métricas en Test Set ({trainer.best_model_name}) ===')
for k, v in test_metrics.items():
    marker = ' ⭐ PRIORITARIA' if k == 'recall' else ''
    print(f'  {k:<12} : {v:.4f}{marker}')

In [ ]:
# Curva ROC
evaluator.plot_roc_curve({trainer.best_model_name: trainer.best_model},
                          trainer.X_test, trainer.y_test)

# Matriz de confusión
evaluator.plot_confusion_matrix(trainer.best_model, trainer.X_test,
                                 trainer.y_test, trainer.best_model_name)

> 💡 **Leyendo la Matriz de Confusión:**
> - **Verdaderos Positivos (VP):** Churners detectados correctamente ✅
> - **Falsos Negativos (FN):** Churners que el modelo NO detectó ❌ ← los más costosos
> - **Falsos Positivos (FP):** Activos que el modelo marcó como churn → costo bajo (campaña innecesaria)
> - **Verdaderos Negativos (VN):** Activos correctamente clasificados ✅

---
## 📈 6. Análisis del Threshold de Decisión <a id='6'></a>

> Por defecto, `predict()` usa un threshold de **0.5**. Pero es posible ajustarlo para optimizar un objetivo de negocio:  
> - Bajar el threshold → más churners detectados (mayor Recall, menor Precision)  
> - Subir el threshold → menos falsos positivos (mayor Precision, menor Recall)

In [ ]:
from sklearn.metrics import precision_recall_curve

y_proba = trainer.best_model.predict_proba(trainer.X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(trainer.y_test, y_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-10)
optimal_idx = f1_scores.argmax()
optimal_threshold = thresholds[optimal_idx]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, precision[:-1], label='Precision', color=COLORS['accent'], lw=2)
ax.plot(thresholds, recall[:-1], label='Recall (detectar churners)', color=COLORS['danger'], lw=2.5)
ax.plot(thresholds, f1_scores[:-1], label='F1 Score', color=COLORS['primary'], lw=2, ls='--')
ax.axvline(optimal_threshold, color='black', lw=1.5, ls=':',
           label=f'Threshold óptimo F1: {optimal_threshold:.2f}')
ax.axvline(0.5, color='gray', lw=1, ls='--', alpha=0.5, label='Threshold default: 0.5')
ax.fill_between(thresholds, recall[:-1], alpha=0.05, color=COLORS['danger'])
ax.set_xlabel('Threshold de Decisión')
ax.set_ylabel('Valor de la Métrica')
ax.set_title('Precision, Recall y F1 vs Threshold de Decisión', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('reports/figures/modeling/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'📌 Threshold óptimo (max F1)  : {optimal_threshold:.3f}')
print(f'   Recall en threshold óptimo : {recall[optimal_idx]:.4f}')
print(f'   Precision en threshold ópt.: {precision[optimal_idx]:.4f}')
print(f'\n💡 Config actual: threshold = 0.5 (default)')
print(f'   Para maximizar detección de churners, considera threshold = {optimal_threshold:.2f}')

---
## 💾 7. Guardar el Modelo Champion <a id='7'></a>

In [ ]:
from src.modeling.registry import ModelRegistry

registry = ModelRegistry()
version_dir = registry.save(
    model=trainer.best_model,
    model_name=trainer.best_model_name,
    features=features_available,
    metrics=test_metrics,
    make_champion=True
)

# Generar metrics.md
metrics_md = evaluator.generate_metrics_markdown(trainer.cv_results, test_metrics)
Path('reports/metrics.md').write_text(metrics_md, encoding='utf-8')

print(f'✅ Modelo champion guardado en: {version_dir}')
print(f'✅ reports/metrics.md actualizado')

---
## ✅ 8. Conclusiones <a id='8'></a>

In [ ]:
print('╔══════════════════════════════════════════════════════╗')
print('║    🤖  RESULTADOS DE MODELADO                       ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'  Modelo champion  : {trainer.best_model_name}')
for k, v in test_metrics.items():
    star = ' ⭐' if k == 'recall' else ''
    print(f'  {k:<13} : {v:.4f}{star}')
print('╠══════════════════════════════════════════════════════╣')
print('  📁 Outputs:')
print('  • models/champion/   (modelo + scaler + info)')
print('  • reports/metrics.md (reporte de métricas)')
print('  • reports/figures/modeling/ (curva ROC, CM, threshold)')
print('  → Siguiente: 05_shap_interpretability.ipynb')
print('╚══════════════════════════════════════════════════════╝')

<div style="background: #fff4f0; border-left: 5px solid #d62828; padding: 15px 20px; border-radius: 6px; margin-top: 20px;">
  <strong>✅ Modelado completado.</strong> El modelo champion ha sido versionado y guardado.<br><br>
  Continúa en <strong>05_shap_interpretability.ipynb</strong> para entender <em>por qué</em> el modelo predice churn para cada cliente.
</div>